# Plankton, Aerosol, ocean Color and Ecosystem (PACE) Satellite Dataset Generation
In this notebook, we'll make csv files for OCI data based on 8D 

In [1]:
!pip install cmocean

In [2]:
!pip install earthaccess

In [3]:
!pip install xarray-datatree

In [4]:
# First, import all necessary libraries, modules, and packages. Earthaccess is essential since we will need this to pull PACE data for our analyses
import earthaccess
import xarray as xr
from datatree import open_datatree
from xarray.backends.api import open_datatree
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
import matplotlib.animation as animation
import os
from PIL import Image, ImageEnhance
import cmocean
import scipy.stats as scipy
import pandas as pd
import matplotlib.patches as mpatches

# OCI -- 8D
Some variables:
1. PACE_OCI_L2_BGC: Lots of products under this - PACE_OCI_L2_BGC_PACE_OCI.20250501T234515.L2.OC_BGC.V3_0.nc_3.0
2. PACE_OCI_L2_AOP
3. PACE_OCI_L3M_AVW
4. PACE_OCI_L3M_POC
5. PACE_OCI_L3M_PAR
6. PACE_OCI_L3M_CHL
7. PACE_OCI_L3M_FLH
8. PACE_OCI_L3M_IOP
9. PACE_OCI_L3M_KD: PACE_OCI.20240305_20240312.L3m.8D.KD.V3_0.Kd.4km.nc **OR** PACE_OCI.20240305.L3m.DAY.KD.V3_0.Kd.0p1deg.nc (coraser just keep in mind)

In [9]:
# Enter your login information for NASA Earthdata
auth = earthaccess.login(persist=True)
# Search the dataset for an instrument you want to work with. In this case, we are going to start with the Ocean Color Instrument (OCI)
results = earthaccess.search_datasets(instrument="oci")
# For an index (item) in results (the dataset), print the associated products (the summary)
    # In this, you will see the products available from your selected instrument
for item in results:
    summary = item.summary()
    print(summary["short-name"])

Enter your Earthdata Login username:  sgryskew
Enter your Earthdata password:  ········


PACE_OCI_L1B_SCI
PACE_OCI_L2_AOP
PACE_OCI_L2_BGC
PACE_OCI_L2_SFREFL
PACE_OCI_L0_SCI
PACE_OCI_L1A_SCI
PACE_OCI_L1C_SCI
PACE_OCI_L2_AER_UAA
PACE_OCI_L2_AER_UAA_NRT
PACE_OCI_L2_AOP_NRT
PACE_OCI_L2_AOP_NRT
PACE_OCI_L2_BGC_NRT
PACE_OCI_L2_BGC_NRT
PACE_OCI_L2_CLOUD
PACE_OCI_L2_CLOUD_MASK
PACE_OCI_L2_CLOUD_MASK_NRT
PACE_OCI_L2_CLOUD_MASK_NRT
PACE_OCI_L2_CLOUD_NRT
PACE_OCI_L2_CLOUD_NRT
PACE_OCI_L2_IOP
PACE_OCI_L2_IOP_NRT
PACE_OCI_L2_IOP_NRT
PACE_OCI_L2_LANDVI
PACE_OCI_L2_LANDVI_NRT
PACE_OCI_L2_LANDVI_NRT
PACE_OCI_L2_PAR
PACE_OCI_L2_PAR_NRT
PACE_OCI_L2_PAR_NRT
PACE_OCI_L2_SFREFL_NRT
PACE_OCI_L2_SFREFL_NRT
PACE_OCI_L2_UVAI_UAA
PACE_OCI_L2_UVAI_UAA_NRT
PACE_OCI_L3B_AVW
PACE_OCI_L3B_AVW_NRT
PACE_OCI_L3B_AVW_NRT
PACE_OCI_L3B_CARBON
PACE_OCI_L3B_CARBON_NRT
PACE_OCI_L3B_CARBON_NRT
PACE_OCI_L3B_CHL
PACE_OCI_L3B_CHL_NRT
PACE_OCI_L3B_CHL_NRT
PACE_OCI_L3B_FLH
PACE_OCI_L3B_FLH_NRT
PACE_OCI_L3B_FLH_NRT
PACE_OCI_L3B_IOP
PACE_OCI_L3B_IOP_NRT
PACE_OCI_L3B_IOP_NRT
PACE_OCI_L3B_KD
PACE_OCI_L3B_KD_NRT
PACE_OCI_L

In [11]:
## Anyway, let's proceed. Alter the time span (tspan) to your frame of interest
tspan = ("2024-01-01", "2025-06-30") #"2024-02-05"  # if you just want one day, just do the tspan with the same begin and end date
lon_min, lon_max = -128.045, -120.517 #og
lat_min, lat_max = 25.179, 31.164 #og 
bbox = (lon_min, lat_min, lon_max, lat_max)
shortname = "PACE_OCI_L3M_CHL"#PACE_OCI_L3M_IOP
clouds = (0,20) #change to (0,100) to see all satellite images (might have clear spot over spot of interest, even if cloudy)

In [13]:
results = earthaccess.search_data(
    short_name=shortname,
    granule_name= '*.8D*4km*',#"*DAY*RRS*4km*.nc", #'*.D*4km*',   #Kd:"*4km*.nc",  #*.8D*4km*", #"*DAY*4km*.nc*" #"*AOP*.nc*"
    temporal=tspan,
    bounding_box=bbox
)
paths = earthaccess.open(results)

QUEUEING TASKS | :   0%|          | 0/53 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/53 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/53 [00:00<?, ?it/s]

In [14]:
index=1 #17=13 # set the day you want to look at
datatree = open_datatree(paths[index]) # open the dataset based on that day
dataset = xr.merge(datatree.to_dict().values()) # merge the datatree to a dictionary format
dataset = dataset.set_coords(("lon", "lat")) # set the map based on lat and lon
dataset
# NIR BANDS (780 nm to 2500 nm): 779 nm, 784 nm, 789 nm, ... 2258 nm # see PACE_OCI_L3M_SFREFL for them

C:\Users\Owner\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


<xarray.Dataset> Size: 149MB
Dimensions:  (lat: 4320, lon: 8640, rgb: 3, eightbitcolor: 256)
Coordinates:
  * lat      (lat) float32 17kB 89.98 89.94 89.9 89.85 ... -89.9 -89.94 -89.98
  * lon      (lon) float32 35kB -180.0 -179.9 -179.9 ... 179.9 179.9 180.0
Dimensions without coordinates: rgb, eightbitcolor
Data variables:
    chlor_a  (lat, lon) float32 149MB ...
    palette  (rgb, eightbitcolor) uint8 768B ...
Attributes: (12/64)
    product_name:                      PACE_OCI.20240313_20240320.L3m.8D.CHL....
    instrument:                        OCI
    title:                             OCI Level-3 Standard Mapped Image
    project:                           Ocean Biology Processing Group (NASA/G...
    platform:                          PACE
    source:                            satellite observations from OCI-PACE
    ...                                ...
    identifier_product_doi:            10.5067/PACE/OCI/L3M/CHL/3.0
    keywords:                          Earth Science > Oceans > Ocean Chemist...
    keywords_vocabulary:               NASA Global Change Master Directory (G...
    data_bins:                         11166039
    data_minimum:                      0.0012108864
    data_maximum:                      99.83112

In [15]:
# Make some arrays for data to be appended to
# time relative
start_date = []
start_year = []
start_month = []
start_day = []
end_date = []
end_year = []
end_month = []
end_day = []

In [16]:
# FOR OCI:
# statistical lists
chl_mean = [] 
chl_geometric_mean = []
chl_std = []
chl_geometric_std = []
chl_N = []
chl_median = []
chl_var = []

In [ ]:
for i in range(0,len(paths)):
    datatree = open_datatree(paths[i]) # open the dataset based on that day
    dataset = xr.merge(datatree.to_dict().values()) # merge the datatree to a dictionary format
    dataset = dataset.set_coords(("lon", "lat")) # set the map based on lat and lon # change to lon and lat for L3 data
    # print the file info
    print(f'INDEX NUMBER: {i}')
    print(dataset.attrs["product_name"])

    # PACE_OCI.20240305_20240312.L3m.8D.KD.V3_0.Kd.4km.nc
    # extract the year, month, day of the file
    str_file = str(paths[i].key)     
    date_start = str_file[9:17] 
    ys, ms, ds = str_file[9:13],str_file[13:15],str_file[15:17]
    print(f'{ys}, {ms}, {ds}')
    date_end = str_file[18:26]
    ye, me, de = str_file[18:22],str_file[22:24],str_file[24:26]
    print(f'{ye}, {me}, {de}')
    print(date_end)
    # create a dataset that selects the points in my ROI
    dataset_masked = dataset.sel(lon=slice(bbox[0], bbox[2]),lat=slice(bbox[3], bbox[1]))
    
    # work with the dataset to get chlor_a cleaned up with no nan values
    #kd = dataset_masked['Kd'].sel(wavelength=490) # if working w/ multiple wavelengths
    chla = dataset_masked['chlor_a']
    
    # drop lat and lon slices where all values are NaN
    chla_cleaned = chla.dropna(dim='lat', how='all').dropna(dim='lon', how='all')
    
    # flatten to 1D numpy array and remove remaining individual NaNs
    chla_flattened = chla_cleaned.values.flatten()
    chla_extracted = chla_flattened[~np.isnan(kd_flattened)]
    
    # count the number of nans in the array
    num_nans = np.isnan(chla_extracted).sum()
    print(f"Number of nans: {num_nans}") 

    
    # finally, we can start the stat. calculations and move forward since all the data has been processed and cleaned!
    chla_m = np.mean(chla_extracted) 
    chla_geomean = scipy.gmean(chla_extracted) 
    chla_med = np.median(chla_extracted) 
    chla_STD = np.std(chla_extracted) 
    chla_gstd = scipy.gstd(chla_extracted) 
    chla_variance = np.var(chla_extracted) 
    chla_n = chla_extracted.size 
    chla_IQR = scipy.iqr(chla_extracted) 

    # finally, append lists with the statistical values
    start_date.append(date_start)
    start_year.append(ys)
    start_month.append(ms)
    start_day.append(ds)
    end_date.append(date_end)
    end_year.append(ye)
    end_month.append(me)
    end_day.append(de)
    
    chl_mean.append(chla_m) 
    chl_geometric_mean.append(chla_geomean) 
    chl_median.append(chla_med) 
    chl_std.append(chla_STD) 
    chl_geometric_std.append(chla_gstd) 
    chl_var.append(chla_variance) 
    chl_N.append(chla_n) 

In [ ]:
pace = {
    'start date': start_date, 'end date': end_date,
    'start year': start_year, 'start month': start_month, 'start day': start_day,
    'end year': end_year, 'end month': end_month, 'end day': end_day,
    'Chla mean': chl_mean, 
    'Chla geometric mean': chl_geometric_mean,
    'Chla median:': chl_median, 
    'Chla std': chl_std, 
    'Chla geometric std': chl_geometric_std,
    'Chla variance': chl_var, 
    'Chla N': chl_N,
    'Chla IQR': chl_iqr
}
pace_df = pd.DataFrame(data=pace)
pace_df['start date'] = pd.to_datetime(pace_df['start date'].astype(str), format="%Y%m%d")
pace_df['end date'] = pd.to_datetime(pace_df['end date'].astype(str), format="%Y%m%d")
pace_df

In [ ]:
pace_df.to_csv(f'/home/jovyan/Desktop/data/PACE_timeseries_D_{shortname}.csv')

In [ ]:
start_date.clear()
start_year.clear()
start_month.clear()
start_day.clear()
end_date.clear()
end_year.clear()
end_month.clear()
end_day.clear()

# statistical lists
chl_mean.clear()
chl_geometric_mean.clear()
chl_median.clear()
chl_std.clear()
chl_geometric_std.clear()
chl_var.clear()
chl_N.clear()
chl_iqr.clear()